# Université Paul Sabatier
# M1 IAFA - Foundations of Information Retrieval - 2026

Instructors: Lynda Tamine, Karim Radouane and Ahmed Rayane Kebir

Notebook proposed by : Jesús Lovón and José G. Moreno

---

💡 Consider developing auxiliary scripts and functions that will enable you to reuse recurring commands in this practical work (PW) and future ones. This would help you keeping good code practice and make debugging easier.


### Attention❗ About TP grading:
🚨 *Code questions*: Fill in the missing code in the corresponding sections (commented code gets the best marks).

🚨 *Open questions*: Write your textual answer as a comment in the corresponding cells.

🚨 *Keep your outputs*: **Empty outputs (notebook or non-executed cells) correspond to 0 points**.

---

# TP 4. PyTerrier - Neural Re-Ranking

In this PW you will learn to:

 - reclassify documents using neural models such as KNRM, Vanilla BERT, EPIC and monoT5.

## Installations and Setup

> 👉 This PW requires *GPU runtime*

- Installing PyTerrier and  PyTerrier plug-ins  

> We install the PyTerrier plugins [OpenNIR](https://opennir.net/) and [monoT5](https://github.com/terrierteam/pyterrier_t5). You can safely ignore package version errors.

In [ ]:
!pip install -q --upgrade python-terrier
!pip install -q --upgrade git+https://github.com/Georgetown-IR-Lab/OpenNIR
!pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_t5


  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# executer pour la cellule 14
!pip install transformers==4.3.3
!pip install tokenizers==0.22.0


  Using cached transformers-4.3.3-py3-none-any.whl.metadata (36 kB)
  Using cached sacremoses-0.1.1-py3-none-any.whl.metadata (8.3 kB)
  Using cached tokenizers-0.10.3.tar.gz (212 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached transformers-4.3.3-py3-none-any.whl (1.9 MB)
Using cached sacremoses-0.1.1-py3-none-any.whl (897 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


We initialize PyTerrier and import libraries as usual

In [ ]:
import os
import pyterrier as pt
from pyterrier.measures import * # allow for natural measure names



# Init PyTerrier
if not pt.started():
    pt.init()

cord19 = pt.datasets.get_dataset('irds:cord19/trec-covid')



/tmp/ipykernel_15343/385787361.py:8: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_15343/385787361.py:9: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [ ]:
import pandas as pd


You are already familiarized with the indexation for CORD19

In [ ]:
import os
!rm -rf ./terrier_cord19/

pt_index_path = './terrier_cord19'

if not os.path.exists(pt_index_path + "/data.properties"):
    # create the index, using the IterDictIndexer indexer
    indexer = pt.index.IterDictIndexer(pt_index_path, text_attrs=['abstract'], meta=['title','docno'])

    # we give the dataset get_corpus_iter() directly to the indexer
    # while specifying the fields to index and the metadata to record
    # index_ref = indexer.index(cord19.get_corpus_iter(),
    #                           text_attrs=['abstract'])
    indexref = indexer.index(cord19.get_corpus_iter(), )

else:
    # if you already have the index, use it.
    indexref = pt.IndexRef.of(pt_index_path + "/data.properties")

index = pt.IndexFactory.of(indexref)

cord19/trec-covid documents:   0%|          | 0/192509 [00:00<?, ?it/s]

09:08:48.489 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (6iu1dtyl) - further warnings are suppressed
09:09:56.152 [ForkJoinPool-1-worker-1] ERROR org.terrier.structures.indexing.Indexer -- Could not finish MetaIndexBuilder: 
java.io.IOException: Key 8lqzfj2e is not unique: 37597,11755
For MetaIndex, to suppress, set metaindex.compressed.reverse.allow.duplicates=true
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.mergeTwo(FSOrderedMapFile.java:1374)
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.close(FSOrderedMapFile.java:1308)
	at org.terrier.structures.indexing.BaseMetaIndexBuilder.close(BaseMetaIndexBuilder.java:321)
	at org.terrier.structures.indexing.classical.BasicIndexer.indexDocuments(BasicIndexer.java:270)
	at org.terrier.structures.indexing.classical.BasicIndexer.createDirectIndex(BasicIndexer.java:388)
	at org.terrier.structures.indexing.Indexer.index(In

# I. Re-Rankers

Let's start exploring some neural re-ranking methods! We can build them from scratch using `onir_pt.reranker`.

OpenNIR's re-ranking model is composed of :
 - `ranker` (for example, `drmm`, `knrm`, or `pacrr`). This defines the neural architecture for ranking.
 - `vocab` (for example, `wordvec_hash`, or `bert`). This defines how the text is encoded by the model. This approach makes it easy to exchange different text representations.

Running this line will take a few minutes, as it downloads and prepares the word vectors.

In [ ]:
import onir_pt

knrm = onir_pt.reranker('knrm', 'wordvec_hash', text_field='abstract')

Better speed can be achieved with apex installed from https://www.github.com/nvidia/apex.
config file not found: config
[2026-03-29 09:10:13,273][WordvecHashVocab][DEBUG] [starting] reading cached at /root/data/onir/vocab/wordvec_hash/fasttext-wiki-news-300d-1M.p
[2026-03-29 09:10:22,850][WordvecHashVocab][DEBUG] [finished] reading cached at /root/data/onir/vocab/wordvec_hash/fasttext-wiki-news-300d-1M.p [9.58s]


Let's see how these models work for ranking!

Similar to previous PWs, we load the topics (queries) and qrels.

Then, we create train/dev/test splits for CORD19.

At this stage, we evaluate directly our models (without training).


In [ ]:
tfidf = pt.BatchRetrieve(indexref, wmodel="TF_IDF") % 50
get_text = pt.text.get_text(cord19, 'abstract') #>> pt.apply.title_abstract(lambda r: r['title'] + ' ' + r['abstract'])


/tmp/ipykernel_15343/343583861.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  tfidf = pt.BatchRetrieve(indexref, wmodel="TF_IDF") % 50


In [ ]:
topics = cord19.get_topics(variant='description')
qrels = cord19.get_qrels()

In [ ]:
SEED=42

from sklearn.model_selection import train_test_split

tr_va_topics, test_topics = train_test_split(topics, test_size=15, random_state=SEED)
train_topics, valid_topics =  train_test_split(tr_va_topics, test_size=5, random_state=SEED)


test_qrels = qrels # seulement les annotations des topics en réponse sont utilisés, donc pas de problème si on utilise tout
train_qrels = qrels
valid_qrels = qrels

In [ ]:
# build a sub-pipeline to get the concatenated title and abstract text
pipeline = tfidf >> get_text >> knrm
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    qrels,
    names=['TFIDF', 'TFIDF >> KNRM'],
    eval_metrics=[AP(rel=2), nDCG, nDCG@10, P(rel=2)@10]
)

[2026-03-29 09:10:28,904][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:10:29,511][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-03-29 09:10:29,529][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.12/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: TFIDF >> KNRM ((TerrierRetr(TF_IDF) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x79b664aa91f0> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-03-29 09:10:33,197][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:10:33,198][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/188 s<?, ?it/s]

[2026-03-29 09:10:34,256][onir_pt][DEBUG] [finished] batches: [1.06s] [188it] [177.83it/s]


,name,nDCG,nDCG@10,AP(rel=2),P(rel=2)@10
0,TFIDF,0.123589,0.595818,0.054568,0.546667
1,TFIDF >> KNRM,0.112743,0.424182,0.042568,0.360000


In [ ]:
tfidf >> get_text >> knrm

[2026-03-29 09:10:34,365][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:10:34,365][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-03-29 09:10:34,380][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]
[2026-03-29 09:10:34,381][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:10:34,382][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-03-29 09:10:34,403][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]
[2026-03-29 09:10:34,405][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:10:34,405][onir_pt][DEBUG] [starting] batches


...

batches: 0it s, ?it/s]

[2026-03-29 09:10:34,444][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]
[2026-03-29 09:10:34,538][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:10:34,539][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-03-29 09:10:34,553][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


(TerrierRetr(TF_IDF) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x79b664aa91f0> >> onir(knrm,wordvec_hash))

This doesn't work very well because the model is not trained; it uses random weights to combine the similarity matrix scores.

# II. Training the re-ranker

You can train re-ranking models in PyTerrier using the `fit` method.

In [ ]:
pipeline.fit(
    train_topics,
    train_qrels,
    valid_topics,
    valid_qrels)

[2026-03-29 09:10:38,799][onir_pt][DEBUG] [starting] validation
[2026-03-29 09:10:38,800][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:10:38,800][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/63 s<?, ?it/s]

[2026-03-29 09:10:38,951][onir_pt][DEBUG] [finished] batches: s] [63it] [419.13it/s]
[2026-03-29 09:10:39,071][onir_pt][DEBUG] [finished] validation s]
[2026-03-29 09:10:39,072][onir_pt][INFO] pre-validation: 0.0053
[2026-03-29 09:10:39,082][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:10:39,082][onir_pt][DEBUG] [starting] training
[2026-03-29 09:10:39,083][onir_pt][DEBUG] [starting] train pairs


train pairs:   0%|          | 0/1024 s<?, ?it/s]

[2026-03-29 09:10:39,516][onir_pt][DEBUG] not enough negs
[2026-03-29 09:10:39,970][onir_pt][DEBUG] not enough negs
[2026-03-29 09:10:41,033][onir_pt][DEBUG] not enough negs
[2026-03-29 09:10:42,019][onir_pt][DEBUG] not enough negs
[2026-03-29 09:10:42,463][onir_pt][DEBUG] not enough negs
[2026-03-29 09:10:43,417][onir_pt][DEBUG] not enough negs
[2026-03-29 09:10:44,358][onir_pt][DEBUG] [finished] train pairs: [5.28s] [1024it] [194.12it/s]
[2026-03-29 09:10:44,359][onir_pt][DEBUG] [finished] training [5.28s]
[2026-03-29 09:10:44,359][onir_pt][INFO] training   it=0 loss=0.2451
[2026-03-29 09:10:44,360][onir_pt][DEBUG] [starting] validation
[2026-03-29 09:10:44,360][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:10:44,362][onir_pt][DEBUG] [starting] batches


...

[2026-03-29 09:12:53,744][onir_pt][DEBUG] [finished] batches: s] [63it] [342.09it/s]
[2026-03-29 09:12:53,875][onir_pt][DEBUG] [finished] validation s]
[2026-03-29 09:12:53,876][onir_pt][INFO] validation it=22 map=0.0060 ndcg=0.0154 P_10=0.0700
[2026-03-29 09:12:53,876][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:12:53,877][onir_pt][DEBUG] [starting] training
[2026-03-29 09:12:53,877][onir_pt][DEBUG] [starting] train pairs


train pairs:   0%|          | 0/1024 s<?, ?it/s]

[2026-03-29 09:12:54,430][onir_pt][DEBUG] not enough negs
[2026-03-29 09:12:54,769][onir_pt][DEBUG] not enough negs
[2026-03-29 09:12:55,092][onir_pt][DEBUG] not enough negs
[2026-03-29 09:12:55,821][onir_pt][DEBUG] not enough negs
[2026-03-29 09:12:56,324][onir_pt][DEBUG] not enough negs
[2026-03-29 09:12:58,620][onir_pt][DEBUG] not enough negs
[2026-03-29 09:12:59,200][onir_pt][DEBUG] not enough negs
[2026-03-29 09:12:59,486][onir_pt][DEBUG] not enough negs
[2026-03-29 09:12:59,496][onir_pt][DEBUG] [finished] train pairs: [5.62s] [1024it] [182.25it/s]
[2026-03-29 09:12:59,497][onir_pt][DEBUG] [finished] training [5.62s]
[2026-03-29 09:12:59,497][onir_pt][INFO] training   it=23 loss=0.2234
[2026-03-29 09:12:59,498][onir_pt][DEBUG] [starting] validation
[2026-03-29 09:12:59,498][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:12:59,499][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/63 s<?, ?it/s]

[2026-03-29 09:12:59,706][onir_pt][DEBUG] [finished] batches: s] [63it] [305.50it/s]
[2026-03-29 09:12:59,913][onir_pt][DEBUG] [finished] validation s]
[2026-03-29 09:12:59,914][onir_pt][INFO] validation it=23 map=0.0060 ndcg=0.0153 P_10=0.0700
[2026-03-29 09:12:59,914][onir_pt][INFO] early stopping; model reverting back to it=3


In [ ]:
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    qrels,
    names=['TFIDF', 'TFIDF >> KNRM (trained)'],
    eval_metrics=[AP(rel=2), nDCG, nDCG@10, P(rel=2)@10]
)

[2026-03-29 09:12:59,968][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:12:59,969][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-03-29 09:12:59,994][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.12/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: TFIDF >> KNRM (trained) ((TerrierRetr(TF_IDF) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x79b664aa91f0> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-03-29 09:13:01,875][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:13:01,876][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/188 s<?, ?it/s]

[2026-03-29 09:13:02,407][onir_pt][DEBUG] [finished] batches: s] [188it] [353.76it/s]


,name,nDCG,nDCG@10,AP(rel=2),P(rel=2)@10
0,TFIDF,0.123589,0.595818,0.054568,0.546667
1,TFIDF >> KNRM (trained),0.121854,0.521672,0.052443,0.453333


#### Question ✍
1. The result is higher, but still not as good as TFIDF. Propose a hypothesis on the problem.

The results show that the neural re-ranker (KNRM) is actually performing slightly worse than the simple keyword-based TF-IDF across all metrics (AP, P@10, nDCG).

We have:

|  name | nDGC  | nDGC@10  | AP(rel=2)  | P(rel=2)@10 |
|---|---|---|---|---|
| TFIDF | 0.123  | 0.595 | 0.054 | 0.546  |
|  TFIDS>>KNRM(trained) | 0.121 | 0.521  | 0.052 | 0.453 |
|  TFIDS>>KNRM  | 0.112  |  0.424 | 0.042  | 0.360  |

We can pretty clearly see that the result is higher but not quite as high as the TFIDF result.

This can be explained by the following reasons:

- Since the KNRM model is kernel based it relies on pre-training. the covid19 dataset being pretuy domain specific. It doesn't give very optimal results.
- Following the domain specific spectrum. We can also say that since TF-IDF matches the exact vocabulary and Kernel Pooling doesn't. This can be a factor to the poor performance of our pipeline.




# III. Vanilla BERT

Contextualized language models, such as [BERT](https://arxiv.org/abs/1810.04805), are much more powerful neural models that have proven effective for classification.

We will try to use a “vanilla” (or “mono”) version of the BERT model. The BERT model is pre-trained for language modeling and next-sentence prediction.

In [ ]:
del knrm # clear out memory from KNRM
vbert = onir_pt.reranker('vanilla_transformer', 'bert', text_field='abstract', vocab_config={'train': True})

Let's see how this model performs on the TREC CORD19 collection.

In [ ]:
pipeline = tfidf % 50 >> get_text >> vbert
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    qrels,
    names=['TFIDF', 'TFIDF >> VBERT'],
    baseline=0,
    eval_metrics=[AP(rel=2), nDCG, nDCG@10, P(rel=2)@10]
)

[2026-03-29 09:13:09,063][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:13:09,209][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-03-29 09:13:09,221][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.12/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: TFIDF >> VBERT ((TerrierRetr(TF_IDF) >> RankCutoff(50) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x79b664aa91f0> >> onir(vanilla_transformer,bert)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-03-29 09:13:10,677][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:13:10,680][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/188 s<?, ?it/s]

[2026-03-29 09:13:37,614][onir_pt][DEBUG] [finished] batches: [26.93s] [188it] [ 6.98it/s]


,name,nDCG,nDCG@10,AP(rel=2),P(rel=2)@10,nDCG +,nDCG -,nDCG p-value,nDCG@10 +,nDCG@10 -,nDCG@10 p-value,AP(rel=2) +,AP(rel=2) -,AP(rel=2) p-value,P(rel=2)@10 +,P(rel=2)@10 -,P(rel=2)@10 p-value
0,TFIDF,0.123589,0.595818,0.054568,0.546667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TFIDF >> VBERT,0.121068,0.526283,0.051388,0.466667,4.0,11.0,0.537761,6.0,9.0,0.200217,4.0,11.0,0.249944,5.0,8.0,0.110861


Added by Henry JIMENEZ:
Here we can't really say that VBERT model is outperforming TFIDF. All this complexity added proves unefective. Maybe SLM's could be more effective ?
This reminds me. Maybe a TER subject could talk about how can SLM's can level or even outperform LLM in domain specific tasks ?

As we can see, although the model is pre-trained, it doesn't perform very well. This is because it is not tuned for the relevance ranking task.

However, we can train the model for ranking (as described above for KNRM).

# IV. monoT5

The [monoT5 model](https://arxiv.org/abs/2003.06713) evaluates documents using a causal language model. Let's see how this approach works on TREC COVID.

The `MonoT5ReRanker` class in `pyterrier_t5` automatically loads a version of the monoT5 workbook, which is trained on the MS MARCO passage dataset.

In [ ]:
from pyterrier_t5 import MonoT5ReRanker
monoT5 = MonoT5ReRanker(text_field='abstract')

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
tfidf >> get_text >> monoT5

(TerrierRetr(TF_IDF) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x79b664aa91f0> >> MonoT5(castorini/monot5-base-msmarco))

In [ ]:
pipeline = (tfidf >> get_text >> monoT5)
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    qrels,
    names=['TFIDF', 'TFIDF >> T5'],
    eval_metrics=[AP(rel=2), nDCG, nDCG@10, P(rel=2)@10, "mrt"]
)


monoT5:   0%|          | 0/188 s<?, ?batches/s]

,name,nDCG,nDCG@10,AP(rel=2),P(rel=2)@10,mrt
0,TFIDF,0.123589,0.595818,0.054568,0.546667,578.152801
1,TFIDF >> T5,0.128997,0.700101,0.060670,0.606667,31634.993203


Added by Henry JIMENEZ:

Here we can see that the result outperforms TF-IDF by an outstanding 10 %. In all metrics and by an unfathomable amount for mrt. monoT5 proves to be the superior model. Not as much as an SLM though. Maybe a study with specialised benchmarks in a TER could prove this statement ?

As expected, the results are much better in terms of NDCG@10 (0.5958 vs. 0.6855).

# Application

#### Question ✍

Simlar to the previous PW, use the models implemented for cord19 in a question-answer task.

In this context, queries are questions and documents are documents that might contain the answer.

Note that you'll need to redo the indexing as well as the other steps studied in this tutorial. You can download the dataset using the lines of code below.

In [ ]:
import re
from pandas import DataFrame

fiqa = {}
fiqa['train'] = pt.datasets.get_dataset('irds:beir/fiqa/train')
fiqa['valid'] = pt.datasets.get_dataset('irds:beir/fiqa/dev')
fiqa['test'] = pt.datasets.get_dataset('irds:beir/fiqa/test')

def clean_query(topics: DataFrame):
    topics['query'] = topics['query'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))



test_topics = fiqa['test'].get_topics(variant='text')
clean_query(test_topics)
test_qrels = fiqa['test'].get_qrels()

train_topics = fiqa['train'].get_topics(variant='text')
clean_query( train_topics )
train_qrels = fiqa['train'].get_qrels()

valid_topics = fiqa['valid'].get_topics(variant='text')
clean_query( valid_topics )
valid_qrels = fiqa['valid'].get_qrels()

#Dataset pour créer l'index :
dataset_fiqa = pt.datasets.get_dataset('irds:beir/fiqa')



In [ ]:
#### Indexation dataset fiqa

import os
!rm -rf ./terrier_fiqa/

pt_index_path = './terrier_fiqa'

if not os.path.exists(pt_index_path + "/data.properties"):
    # create the index, using the IterDictIndexer indexer
    indexer = pt.index.IterDictIndexer(pt_index_path, text_attrs=['text'], meta=['docno'], fields=True)

    # we give the dataset get_corpus_iter() directly to the indexer
    # while specifying the fields to index and the metadata to record
    # index_ref = indexer.index(cord19.get_corpus_iter(),
    #                           text_attrs=['abstract'])
    indexref_piqa = indexer.index(dataset_fiqa.get_corpus_iter(), )

else:
    # if you already have the index, use it.
    indexref_piqa = pt.IndexRef.of(pt_index_path + "/data.properties")

index_piqa = pt.IndexFactory.of(indexref)



beir/fiqa documents:   0%|          | 0/57638 s<?, ?it/s]

09:14:28.824 [ForkJoinPool-2-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (7915) - further warnings are suppressed
09:14:44.554 [ForkJoinPool-2-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 39 empty documents


In [ ]:
tfidf = pt.BatchRetrieve(indexref_piqa, wmodel="TF_IDF") % 50
get_text = pt.text.get_text(dataset_fiqa, 'text') #>> pt.apply.title_abstract(lambda r: r['title'] + ' ' + r['abstract'])

/tmp/ipykernel_15343/344516202.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  tfidf = pt.BatchRetrieve(indexref_piqa, wmodel="TF_IDF") % 50


In [ ]:
knrm = onir_pt.reranker('knrm', 'wordvec_hash', text_field='text')

[2026-03-29 09:14:44,672][WordvecHashVocab][DEBUG] [starting] reading cached at /root/data/onir/vocab/wordvec_hash/fasttext-wiki-news-300d-1M.p
[2026-03-29 09:14:54,400][WordvecHashVocab][DEBUG] [finished] reading cached at /root/data/onir/vocab/wordvec_hash/fasttext-wiki-news-300d-1M.p [9.73s]


In [ ]:
# build a sub-pipeline to get the concatenated title and abstract text
# non trained KNRM
pipeline = tfidf >> get_text >> knrm
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    test_qrels,
    names=['TFIDF', 'TFIDF >> KNRM'],
    # On met nos données à rel=1 puisqu'il n'existe pas un niveau de pertinence égal à 2 dans qrels.
    eval_metrics=[AP(rel=1), nDCG, nDCG@10, P(rel=1)@10]
)

[2026-03-29 09:15:01,037][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:15:01,382][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-03-29 09:15:01,420][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.12/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: TFIDF >> KNRM ((TerrierRetr(TF_IDF) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x79b66662c770> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-03-29 09:15:37,798][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:15:37,799][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/8100 s<?, ?it/s]

[2026-03-29 09:15:56,341][onir_pt][DEBUG] [finished] batches: [18.54s] [8100it] [436.84it/s]


,name,nDCG,nDCG@10,AP(rel=2),P(rel=2)@10
0,TFIDF,0.289927,0.242948,0.0,0.0
1,TFIDF >> KNRM,0.142833,0.052018,0.0,0.0


In [ ]:
pipeline.fit(
    train_topics,
    train_qrels,
    valid_topics,
    valid_qrels)

[2026-03-29 09:22:56,542][onir_pt][DEBUG] [starting] validation
[2026-03-29 09:22:56,543][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:22:56,543][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/6250 s<?, ?it/s]

[2026-03-29 09:23:11,236][onir_pt][DEBUG] [finished] batches: [14.69s] [6250it] [425.40it/s]
[2026-03-29 09:23:11,321][onir_pt][DEBUG] [finished] validation [14.78s]
[2026-03-29 09:23:11,321][onir_pt][INFO] pre-validation: 0.0410
[2026-03-29 09:23:11,334][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:23:11,334][onir_pt][DEBUG] [starting] training
[2026-03-29 09:23:11,334][onir_pt][DEBUG] [starting] train pairs


train pairs:   0%|          | 0/1024 s<?, ?it/s]

[2026-03-29 09:23:17,666][onir_pt][DEBUG] [finished] train pairs: [6.33s] [1024it] [161.73it/s]
[2026-03-29 09:23:17,667][onir_pt][DEBUG] [finished] training [6.33s]
[2026-03-29 09:23:17,668][onir_pt][INFO] training   it=0 loss=0.2479
[2026-03-29 09:23:17,668][onir_pt][DEBUG] [starting] validation
[2026-03-29 09:23:17,668][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 09:23:17,669][onir_pt][DEBUG] [starting] batches


...

[2026-03-29 10:01:03,768][onir_pt][DEBUG] [finished] batches: [16.56s] [6250it] [377.45it/s]
[2026-03-29 10:01:03,857][onir_pt][DEBUG] [finished] validation [16.65s]
[2026-03-29 10:01:03,858][onir_pt][INFO] validation it=102 map=0.2044 ndcg=0.2903 P_10=0.0644
[2026-03-29 10:01:03,858][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 10:01:03,858][onir_pt][DEBUG] [starting] training
[2026-03-29 10:01:03,858][onir_pt][DEBUG] [starting] train pairs


train pairs:   0%|          | 0/1024 s<?, ?it/s]

[2026-03-29 10:01:09,943][onir_pt][DEBUG] [finished] train pairs: [6.08s] [1024it] [168.30it/s]
[2026-03-29 10:01:09,944][onir_pt][DEBUG] [finished] training [6.09s]
[2026-03-29 10:01:09,946][onir_pt][INFO] training   it=103 loss=0.1118
[2026-03-29 10:01:09,946][onir_pt][DEBUG] [starting] validation
[2026-03-29 10:01:09,946][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 10:01:09,947][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/6250 s<?, ?it/s]

[2026-03-29 10:01:26,853][onir_pt][DEBUG] [finished] batches: [16.91s] [6250it] [369.69it/s]
[2026-03-29 10:01:26,944][onir_pt][DEBUG] [finished] validation [17.00s]
[2026-03-29 10:01:26,945][onir_pt][INFO] validation it=103 map=0.2090 ndcg=0.2940 P_10=0.0636
[2026-03-29 10:01:26,945][onir_pt][INFO] early stopping; model reverting back to it=83


In [ ]:
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    test_qrels,
    names=['TFIDF', 'TFIDF >> KNRM (trained)'],
    eval_metrics=[AP(rel=1), nDCG, nDCG@10, P(rel=1)@10]
)

[2026-03-29 10:11:31,059][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 10:11:31,060][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-03-29 10:11:31,085][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.12/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: TFIDF >> KNRM (trained) ((TerrierRetr(TF_IDF) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x79b66662c770> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-03-29 10:12:10,659][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 10:12:10,660][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/8100 s<?, ?it/s]

[2026-03-29 10:12:27,899][onir_pt][DEBUG] [finished] batches: [17.24s] [8100it] [469.88it/s]


,name,AP,P@10,nDCG,nDCG@10
0,TFIDF,0.197242,0.067901,0.289927,0.242948
1,TFIDF >> KNRM (trained),0.191779,0.065432,0.285757,0.238424


Vanilla Bert


In [ ]:
del knrm # clear out memory from KNRM
vbert = onir_pt.reranker('vanilla_transformer', 'bert', text_field='text', vocab_config={'train': True})

In [ ]:
pipeline = tfidf % 50 >> get_text >> vbert
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    test_qrels,
    names=['TFIDF', 'TFIDF >> VBERT'],
    baseline=0,
    eval_metrics=[AP(rel=1), nDCG, nDCG@10, P(rel=1)@10]
)

[2026-03-29 10:13:05,405][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 10:13:05,558][onir_pt][DEBUG] [starting] batches


batches: 0it s, ?it/s]

[2026-03-29 10:13:05,575][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.12/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: TFIDF >> VBERT ((TerrierRetr(TF_IDF) >> RankCutoff(50) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x79b66662c770> >> onir(vanilla_transformer,bert)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-03-29 10:13:39,839][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-29 10:13:39,843][onir_pt][DEBUG] [starting] batches


batches:   0%|          | 0/8100 s<?, ?it/s]

[2026-03-29 10:34:42,702][onir_pt][DEBUG] [finished] batches: [21:03] [8100it] [ 6.41it/s]


,name,AP,P@10,nDCG,nDCG@10,AP +,AP -,AP p-value,P@10 +,P@10 -,P@10 p-value,nDCG +,nDCG -,nDCG p-value,nDCG@10 +,nDCG@10 -,nDCG@10 p-value
0,TFIDF,0.197242,0.067901,0.289927,0.242948,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TFIDF >> VBERT,0.028694,0.010185,0.128566,0.021958,62.0,372.0,7.765113e-46,22.0,282.0,2.408661e-49,62.0,372.0,1.403861e-53,23.0,306.0,8.552556e-56


Mono T5

In [ ]:
monoT5 = MonoT5ReRanker(text_field='text')

In [ ]:
pipeline = (tfidf >> get_text >> monoT5)
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    test_qrels,
    names=['TFIDF', 'TFIDF >> T5'],
    eval_metrics=[AP(rel=1), nDCG, nDCG@10, P(rel=1)@10, "mrt"]
)

monoT5:   0%|          | 0/8100 s<?, ?batches/s]

,name,AP,P@10,nDCG,nDCG@10,mrt
0,TFIDF,0.197242,0.067901,0.289927,0.242948,1.828712e+04
1,TFIDF >> T5,0.300543,0.092593,0.380061,0.362656,1.234975e+06


#### Question ✍
Do you see any advantages compared to Learning to Rank?

Neural Learning Models such as Bert and monoT5 have best performances than TFIDF. We see that Bert surpases TF-IDF by 0.1. This outperferformance goes at the cost of inference and training time. This PW used a total of 11.7 GB of RAM, 6 GB Of GPU and 49.4 GB of disk using the T4 GPU provided by Google Collab. Neuronal Network are more precise at the cost of ressources and compute time.